# Activity Cycle Diagram Little's Law Verification 

This notebook demonstrates:
1. Defining an **Activity Cycle Diagram (ACD)** model using JSON specification
2. Converting JSON to SimASM using the **ACD converter**
3. Running a **Little's Law** verification experiment

## Activity Cycle Diagram Formalism

The ACD formalism (Tocher 1960, Carrie 1988) models systems using:
- **Passive states (Queues)**: Ovals where tokens wait
- **Active states (Activities)**: Rectangles where tokens are processed
- **Tokens**: Entities flowing through the diagram

The **Three-Phase Approach**:
- **A-phase**: Time advance (pop next BTO event)
- **B-phase**: Scan activities for start conditions
- **C-phase**: Execute activity completions

In [1]:
# Install simasm (uncomment in Colab)
# !pip install simasm

import simasm
print(f"SimASM version: {simasm.__version__}")

SimASM version: 0.2.8


## 1. Define ACD JSON Specification

The ACD uses an **Activity Transition Table** format:

| Activity | Priority | At-Begin Condition | At-Begin Action | BTO Event | At-End Arcs |
|----------|----------|-------------------|-----------------|-----------|-------------|
| Create   | 1        | marking(C) >= 1   | C--             | Created   | C++ <- creator, Q++ <- new Job |
| Serve    | 2        | marking(S) >= 1 and marking(Q) >= 1 | S--, Q-- | Served | S++ <- server, Jobs++ <- job |

In [2]:
# Define the M/M/5 ACD as JSON
mm5_acd_json = {
  "model_name": "mm5_acd",
  "description": "M/M/5 Queue using Activity Cycle Diagram formalism.Based on Tocher (1960), Carrie (1988) Activity Transition Table format",
  "parameters": {
    "num_servers": {"type": "Nat", "value": 5, "description": "Number of servers (N in M/M/N)"},
    "iat_mean": {"type": "Real", "value": 1.25, "description": "Inter-arrival time mean (1/lambda)"},
    "ist_mean": {"type": "Real", "value": 1.0, "description": "Service time mean (1/mu)"},
    "sim_end_time": {"type": "Real", "value": 1000.0, "description": "Simulation end time"}
  },

  "token_types": {
    "Job": {
      "parent": "Token",
      "attributes": {
        "arrival_time": "Real",
        "service_start_time": "Real"
      },
      "description": "Customer job token"
    },
    "Resource": {
      "parent": "Token",
      "description": "Reusable resource token (creator, server)"
    }
  },

  "queues": {
    "C": {"initial_marking": 1, "token_type": "Resource", "is_resource": True, "description": "Creator resource queue"},
    "Q": {"initial_marking": 0, "token_type": "Job", "is_resource": False, "description": "Job waiting queue"},
    "S": {"initial_marking": 5, "token_type": "Resource", "is_resource": True, "description": "Server resource queue"},
    "Jobs": {"initial_marking": 0, "token_type": "Job", "is_resource": False, "description": "Completed jobs (sink)"}
  },

  "activities": [
    {
      "name": "Create",
      "priority": 1,
      "description": "Job creation activity (bound activity)",
      "at_begin": {
        "condition": "marking(C) >= 1",
        "action": "C--",
        "bind": ["creator_token:C"]
      },
      "bto_event": {
        "time": "interarrival_time",
        "name": "Created"
      },
      "at_end": [
        {"arc": 1, "action": "C++ <- creator_token", "influences": ["Create"]},
        {"arc": 2, "action": "Q++ <- new Job", "influences": ["Serve"]}
      ]
    },
    {
      "name": "Serve",
      "priority": 2,
      "description": "Service activity (conditional activity)",
      "at_begin": {
        "condition": "marking(S) >= 1 and marking(Q) >= 1",
        "action": "S--; Q--",
        "bind": ["server_token:S", "job_token:Q"]
      },
      "bto_event": {
        "time": "service_time",
        "name": "Served"
      },
      "at_end": [
        {"arc": 1, "action": "S++ <- server_token; Jobs++ <- job_token", "influences": ["Serve"]}
      ]
    }
  ],

  "random_streams": {
    "interarrival_time": {
      "distribution": "exponential",
      "params": {"mean": "iat_mean"},
      "stream_name": "arrivals"
    },
    "service_time": {
      "distribution": "exponential",
      "params": {"mean": "ist_mean"},
      "stream_name": "service"
    }
  },

  "observables": {
    "queue_count": {"expression": "marking(Q)", "description": "Number in queue"},
    "servers_busy": {"expression": "num_servers - marking(S)", "description": "Number of busy servers"},
    "in_system": {"expression": "marking(Q) + (num_servers - marking(S))", "description": "Total in system"}
  }
}

# Save JSON to file for the converter
import json
with open("mm5_acd.json", "w") as f:
    json.dump(mm5_acd_json, f, indent=2)
print("Saved mm5_acd.json")

Saved mm5_acd.json


## 2. Convert JSON to SimASM Model

Use the **ACD converter** to generate SimASM code.

The converter implements the **Activity Scanning Algorithm**:
1. **B-phase**: Scan activities in priority order, start one if conditions met
2. **A-phase**: If no activity started, advance time to next BTO event
3. **C-phase**: Execute the BTO event (activity completion)
4. Return to B-phase

In [ ]:
%%simasm convert

convert mm5_acd:
    source: "mm5_acd.json"
    formalism: acd
    register: "mm5_acd"
    print: true
endconvert

## 3. Run Little's Law Experiment

For M/M/5 with λ=0.8, μ=1.0:
- ρ = λ/(5μ) = 0.16 (per server utilization)
- L_s = 5ρ = 0.8 (average busy servers)

In [4]:
%%simasm experiment
// Little's Law Experiment for ACD M/M/5 Queue

experiment LittlesLawACD:
    model := "mm5_acd"
    
    replication:
        count: 30
        warm_up_time: 100.0
        run_length: 1000.0
        seed_strategy: "incremental"
        base_seed: 12345
    endreplication
    
    statistics:
        // L: Average number in system (queue + in service)
        stat L_system: time_average
            expression: "marking(Q) + (num_servers - marking(S))"
        endstat
        
        // L_q: Average number in queue
        stat L_queue: time_average
            expression: "marking(Q)"
        endstat
        
        // L_s: Average number in service (busy servers)
        stat L_service: time_average
            expression: "num_servers - marking(S)"
        endstat

        // Server utilization (fraction of capacity used)
        stat rho_utilization: time_average
            expression: "(num_servers - marking(S)) / num_servers"
        endstat

        // Total departures
        stat total_departures: count
            expression: "marking(Jobs)"
        endstat

        // W: Average time in system
        stat W_system_avg: count
            expression: "total_sojourn_time / departure_count"
        endstat
        
    endstatistics
    
    output:
        format: "json"
        file_path: "acd_littles_law_results.json"
    endoutput
endexperiment

  Replication 30/30...


Statistic,Mean,Std Dev,95% CI
L_system,0.8054,0.0389,"[0.7908, 0.8199]"
L_queue,0.0005,0.0008,"[0.0002, 0.0008]"
L_service,0.8049,0.0389,"[0.7904, 0.8194]"
rho_utilization,0.1610,0.0078,"[0.1581, 0.1639]"
total_departures,799.5000,18.7318,"[792.5062, 806.4938]"
W_system_avg,1.0049,0.0389,"[0.9904, 1.0194]"


ExperimentResult(config=ExperimentConfig(name='LittlesLawACD', model_path='C:\\Users\\steve\\AppData\\Local\\Temp\\simasm_wywy0_7r\\mm5_acd.simasm', model_source=None, replications=ReplicationSettings(count=30, warmup=100.0, length=1000.0, base_seed=12345), statistics=[StatisticConfig(name='L_system', type='time_average', domain=None, expr='marking(Q) + (num_servers - marking(S))', interval=None, condition=None, aggregation='average', start_expr=None, end_expr=None, entity_domain=None), StatisticConfig(name='L_queue', type='time_average', domain=None, expr='marking(Q)', interval=None, condition=None, aggregation='average', start_expr=None, end_expr=None, entity_domain=None), StatisticConfig(name='L_service', type='time_average', domain=None, expr='num_servers - marking(S)', interval=None, condition=None, aggregation='average', start_expr=None, end_expr=None, entity_domain=None), StatisticConfig(name='rho_utilization', type='time_average', domain=None, expr='(num_servers - marking(S)) /

## Summary

This notebook demonstrated:

1. **JSON → SimASM Conversion**: Used the ACD converter to generate executable SimASM code from an Activity Transition Table JSON specification

2. **Three-Phase Algorithm**: The generated code implements the Activity Scanning Algorithm with:
   - B-phase: Scan activities for start conditions
   - A-phase: Time advance to next BTO event
   - C-phase: Execute activity completions

3. **Little's Law Verification**: Confirmed that the generated model produces statistically valid results:
   - Conservation: L = L_q + L_s
   - Little's Law: L = λW

### References

- Tocher, K.D. (1960). The Art of Simulation. English Universities Press.
- Carrie, A.S. (1988). Simulation of Manufacturing Systems. Wiley.
- Little, J.D.C. (1961). A Proof for the Queuing Formula: L = λW. Operations Research.